# 01 - Object / Label Inventory over All Sequence Folders

Stage L0 of the pipeline (`docs/pipeline_restructure_plan.md`). Scans the raw
sequence root folder by folder and produces, under `inventory/<run_id>/`:

- **folder health**: complete / missing files / empty folders,
- **object inventory**: every object name from metadata JSON tags and Label V2
  rows, with class distributions, distance stats and catalog matching,
- **Label V2 profiling**: class values, `NAMEoFFILE` taxonomy, spec columns,
- a raw dump of every Label V2 row (`all_label_v2_rows.csv`).

Leave `SMOKE_TEST = True` (random `SMOKE_LIMIT` folders) to verify, then set it
False for the full scan.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import csv
import json
import random
import re
from collections import Counter, defaultdict
from itertools import islice

import pandas as pd
from tqdm.auto import tqdm

from vision_uss_research.runs import finish_run, new_run_id, start_run

SMOKE_TEST = True
SMOKE_LIMIT = 100
SMOKE_MODE = "random"      # "random" (unbiased) or "first"
SMOKE_SEED = 0
RUN_TAG = "v2"

CATALOG_CSV = REPO_DIR / "configs" / "datasets" / "object_height_catalog.csv"

RUN_ID = new_run_id("inventory", RUN_TAG) + ("_smoke" if SMOKE_TEST else "")
RUN_DIR = ARTIFACTS / "inventory" / RUN_ID
record = start_run(RUN_DIR, config={
    "smoke_test": SMOKE_TEST, "smoke_limit": SMOKE_LIMIT, "smoke_mode": SMOKE_MODE,
    "smoke_seed": SMOKE_SEED, "raw_root": str(RAW_ROOT),
    "catalog_csv": str(CATALOG_CSV)}, repo_root=REPO_DIR)
print(f"run dir: {RUN_DIR}")

In [ ]:
LABEL_OBJECTS_SUBCATEGORY = "label objects"
CLASS_ID_TO_NAME = {"0": "traversable", "1": "low", "2": "unknown", "3": "high"}
VIDEO_EXTENSIONS = {".mp4", ".webm", ".avi"}

def normalize_object_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())

def load_height_catalog(catalog_csv):
    key_to_row = {}
    if not catalog_csv.exists():
        print("WARNING: height catalog not found")
        return key_to_row
    with catalog_csv.open("r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            for name in [row.get("object_name", "")] + str(row.get("aliases", "")).split("|"):
                key = normalize_object_name(name)
                if key:
                    key_to_row.setdefault(key, row)
    return key_to_row

def safe_read_label_csv(path):
    attempts = [{"encoding": "utf-8-sig", "sep": ","}, {"encoding": "utf-8", "sep": ","},
                {"encoding": "latin1", "sep": ","}, {"encoding": "utf-8-sig", "sep": ";"},
                {"encoding": "latin1", "sep": ";"}]
    last_error = None
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            if len(df.columns) > 1:
                return df
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Could not read label CSV: {path}. Last error: {last_error}")

def find_column(columns, candidates):
    norm = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        if cand.strip().lower() in norm:
            return norm[cand.strip().lower()]
    for col in columns:
        for cand in candidates:
            if cand.strip().lower() in str(col).strip().lower():
                return col
    return ""

def extract_json_tags(metadata_path):
    with metadata_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)
    tags = defaultdict(list)
    for tag in metadata.get("tags", []):
        sub = str(tag.get("tagSubCategory", {}).get("name", "")).strip().lower()
        name = str(tag.get("name", "")).strip()
        if sub and name:
            tags[sub].append(name)
    return dict(tags)

def find_sequence_files(seq_dir):
    top_files = [p for p in seq_dir.iterdir() if p.is_file()]
    lower = lambda p: p.name.lower()
    def pick(pred):
        matches = sorted(p for p in top_files if pred(p))
        return matches[0] if matches else None
    return {
        "n_top_files": len(top_files),
        "metadata_json": pick(lambda p: p.suffix.lower() == ".json"),
        "label_v2_csv": pick(lambda p: "pas_m_label_v2" in lower(p) and p.suffix.lower() == ".csv"),
        "front_video": pick(lambda p: "front" in lower(p) and p.suffix.lower() in VIDEO_EXTENSIONS),
        "rear_video": pick(lambda p: "rear" in lower(p) and p.suffix.lower() in VIDEO_EXTENSIONS),
        "topview": pick(lambda p: "topview" in lower(p) and p.suffix.lower() in VIDEO_EXTENSIONS),
    }

def has_any_file(seq_dir, scan_cap=2000):
    return any(p.is_file() for p in islice(seq_dir.rglob("*"), scan_cap))

def classify_folder(found, is_empty):
    if is_empty:
        return "empty"
    if not found["metadata_json"] and not found["label_v2_csv"]:
        return "no_metadata_no_label"
    if not found["metadata_json"]:
        return "missing_metadata_json"
    if not found["label_v2_csv"]:
        return "missing_label_v2"
    if not (found["front_video"] or found["rear_video"]):
        return "missing_camera_video"
    return "complete"

# NAMEoFFILE taxonomy: families like C_highObjectslarge / F_lowObjectssmall,
# categories <L><nn>_<name>; UNC and drive-letter path forms.
FAMILY_RE = re.compile(r"^[A-Z]_\w*[Oo]bjects\w*$")
CATEGORY_RE = re.compile(r"^[A-Z]\d{2}_")

def parse_name_of_file(value):
    parts = [p for p in re.split(r"[\\/]+", str(value)) if p]
    return {
        "tax_family": next((p for p in parts if FAMILY_RE.match(p)), ""),
        "tax_category": next((p for p in parts if CATEGORY_RE.match(p)), ""),
        "tax_type": next((p for p in parts if re.match(r"^Type\d*$", p, re.I)), ""),
    }

In [ ]:
all_folders = sorted(p for p in RAW_ROOT.iterdir() if p.is_dir())
print(f"total folders under raw root : {len(all_folders)}")

if SMOKE_TEST:
    if SMOKE_MODE == "random":
        rng = random.Random(SMOKE_SEED)
        folders = sorted(rng.sample(all_folders, min(SMOKE_LIMIT, len(all_folders))))
    else:
        folders = all_folders[:SMOKE_LIMIT]
else:
    folders = all_folders
print(f"folders to scan              : {len(folders)}")

In [ ]:
catalog = load_height_catalog(CATALOG_CSV)

folder_rows = []
raw_label_rows = []
display_name = {}
json_sequences = defaultdict(set)
label_v2_sequences = defaultdict(set)
label_v2_row_counts = Counter()
class_counts = defaultdict(Counter)
distances_mm = defaultdict(list)
odd_class_examples = defaultdict(set)

for seq_dir in tqdm(folders, desc="Scanning folders", unit="dir"):
    sequence_id = seq_dir.name
    found = find_sequence_files(seq_dir)
    is_empty = found["n_top_files"] == 0 and not has_any_file(seq_dir)
    status = classify_folder(found, is_empty)

    json_objects, v2_objects, errors = [], [], []
    if found["metadata_json"]:
        try:
            tags = extract_json_tags(found["metadata_json"])
            json_objects = tags.get(LABEL_OBJECTS_SUBCATEGORY, [])
        except Exception as e:
            errors.append(f"metadata_json:{type(e).__name__}")
    for name in json_objects:
        key = normalize_object_name(name)
        if key:
            display_name.setdefault(key, name)
            json_sequences[key].add(sequence_id)

    if found["label_v2_csv"]:
        try:
            df = safe_read_label_csv(found["label_v2_csv"])
            columns = list(df.columns)
            object_col = find_column(columns, ["Which object", "object"])
            class_col = find_column(columns, ["class"])
            distance_col = find_column(columns, ["important Distance [mm]", "distance"])
            for _, label_row in df.iterrows():
                rec = {"sequence_id": sequence_id}
                rec.update({str(k): label_row[k] for k in df.columns})
                raw_label_rows.append(rec)
                name = str(label_row[object_col]).strip() if object_col else ""
                if not name or name.lower() == "nan":
                    continue
                key = normalize_object_name(name)
                display_name.setdefault(key, name)
                label_v2_sequences[key].add(sequence_id)
                label_v2_row_counts[key] += 1
                v2_objects.append(name)
                if class_col:
                    class_value = str(label_row[class_col]).strip().split(".")[0]
                    if class_value and class_value.lower() != "nan":
                        class_counts[key][class_value] += 1
                        if class_value not in CLASS_ID_TO_NAME and len(odd_class_examples[class_value]) < 10:
                            odd_class_examples[class_value].add(sequence_id)
                if distance_col:
                    distance = pd.to_numeric(label_row[distance_col], errors="coerce")
                    if pd.notna(distance):
                        distances_mm[key].append(float(distance))
        except Exception as e:
            errors.append(f"label_v2:{type(e).__name__}")

    folder_rows.append({
        "sequence_id": sequence_id, "status": status,
        "n_top_files": found["n_top_files"],
        "has_metadata_json": int(found["metadata_json"] is not None),
        "has_label_v2_csv": int(found["label_v2_csv"] is not None),
        "has_front_video": int(found["front_video"] is not None),
        "has_rear_video": int(found["rear_video"] is not None),
        "has_topview": int(found["topview"] is not None),
        "json_label_objects": "|".join(json_objects),
        "label_v2_objects": "|".join(v2_objects),
        "errors": "|".join(errors),
    })

folders_df = pd.DataFrame(folder_rows)
print(f"scanned {len(folders_df)} folders, {len(raw_label_rows)} Label V2 rows")

## Folder health

In [ ]:
status_summary = folders_df["status"].value_counts().rename_axis("status").to_frame("n_folders")
status_summary["pct"] = (100 * status_summary["n_folders"] / len(folders_df)).round(1)
display(status_summary)

problem_df = folders_df[folders_df["status"] != "complete"]
if len(problem_df):
    display(problem_df[["sequence_id", "status", "n_top_files"]].head(50))

## Object inventory (Label V2 class = ground truth; catalog = height prior)

In [ ]:
object_rows = []
all_keys = sorted(set(json_sequences) | set(label_v2_sequences),
                  key=lambda k: -(len(json_sequences[k] | label_v2_sequences[k])))
for key in all_keys:
    catalog_row = catalog.get(key)
    dists = sorted(distances_mm[key])
    documented = {cid: n for cid, n in class_counts[key].items() if cid in CLASS_ID_TO_NAME}
    if documented:
        majority = max(documented, key=documented.get)
        label_v2_bin = CLASS_ID_TO_NAME[majority]
        purity = round(documented[majority] / sum(documented.values()), 3)
    else:
        label_v2_bin, purity = "", None
    catalog_bin = (catalog_row or {}).get("height_bin", "")
    if not catalog_bin or not label_v2_bin:
        agreement = ""
    elif catalog_bin == "ignore":
        agreement = "ok" if label_v2_bin == "traversable" else "CONFLICT"
    else:
        agreement = "ok" if catalog_bin == label_v2_bin else "CONFLICT"
    object_rows.append({
        "object_name": display_name[key],
        "n_sequences_total": len(json_sequences[key] | label_v2_sequences[key]),
        "n_label_v2_rows": label_v2_row_counts[key],
        "label_v2_class_distribution": "|".join(
            f"{CLASS_ID_TO_NAME.get(c, c)}:{n}" for c, n in sorted(class_counts[key].items())),
        "label_v2_bin": label_v2_bin, "label_v2_bin_purity": purity,
        "catalog_bin_prior": catalog_bin, "bin_agreement": agreement,
        "distance_mm_median": int(dists[len(dists) // 2]) if dists else None,
        "catalog_matched": int(catalog_row is not None),
    })
objects_df = pd.DataFrame(object_rows)
display(objects_df)

if odd_class_examples:
    print("undocumented class values (inspect example sequences):")
    for cv, ex in sorted(odd_class_examples.items()):
        print(f"  class {cv!r}: e.g. {sorted(ex)[:5]}")

## Label V2 profiling (distance gate + taxonomy)

In [ ]:
all_rows_df = pd.DataFrame(raw_label_rows)
DIST_COL = find_column(list(all_rows_df.columns), ["important Distance [mm]", "distance"])
NAMEOF_COL = find_column(list(all_rows_df.columns), ["NAMEoFFILE", "name of file"])

dist = pd.to_numeric(all_rows_df[DIST_COL], errors="coerce").dropna()
pct_2000 = 100 * (dist == 2000).mean() if len(dist) else 0
print(f"distance == 2000 mm: {pct_2000:.1f}% of {len(dist)} rows")
if pct_2000 > 90:
    print(">>> GATE: distance is a spec constant - use CusReplay ObjDist for distance supervision")

tax = all_rows_df[NAMEOF_COL].astype(str).map(parse_name_of_file).apply(pd.Series)
all_rows_df = pd.concat([all_rows_df, tax], axis=1)
fam = all_rows_df[all_rows_df["tax_family"] != ""]
if len(fam):
    display(fam.groupby(["tax_family", "tax_category"])["sequence_id"]
            .nunique().to_frame("n_sequences")
            .sort_values("n_sequences", ascending=False).head(30))

## Save run outputs

In [ ]:
folders_df.to_csv(RUN_DIR / "inventory_folders.csv", index=False)
objects_df.to_csv(RUN_DIR / "inventory_objects.csv", index=False)
all_rows_df.to_csv(RUN_DIR / "all_label_v2_rows.csv", index=False)

finish_run(RUN_DIR, summary={
    "folders_scanned": len(folders_df),
    "folder_status": folders_df["status"].value_counts().to_dict(),
    "unique_objects": len(objects_df),
    "label_v2_rows": len(all_rows_df),
})
print(f"saved to {RUN_DIR}")